In [1]:
import pandas as pd

In [2]:
df = pd.read_json('data/youtube-transcriptions.jsonl', lines=True)

In [3]:
df.head()

,title,published,url,video_id,channel_id,id,text,start,end
0,Training and Testing an Italian BERT - Transfo...,2021-07-06 13:00:03 UTC,https://youtu.be/35Pdoyi6ZoQ,35Pdoyi6ZoQ,UCv83tO5cePwHMt1952IVVHw,35Pdoyi6ZoQ-t0.0,"Hi, welcome to the video.",0.00,9.36
1,Training and Testing an Italian BERT - Transfo...,2021-07-06 13:00:03 UTC,https://youtu.be/35Pdoyi6ZoQ,35Pdoyi6ZoQ,UCv83tO5cePwHMt1952IVVHw,35Pdoyi6ZoQ-t3.0,So this is the fourth video in a Transformers,3.00,11.56
2,Training and Testing an Italian BERT - Transfo...,2021-07-06 13:00:03 UTC,https://youtu.be/35Pdoyi6ZoQ,35Pdoyi6ZoQ,UCv83tO5cePwHMt1952IVVHw,35Pdoyi6ZoQ-t9.36,from Scratch mini series.,9.36,15.84
3,Training and Testing an Italian BERT - Transfo...,2021-07-06 13:00:03 UTC,https://youtu.be/35Pdoyi6ZoQ,35Pdoyi6ZoQ,UCv83tO5cePwHMt1952IVVHw,35Pdoyi6ZoQ-t11.56,"So if you haven't been following along,",11.56,18.48
4,Training and Testing an Italian BERT - Transfo...,2021-07-06 13:00:03 UTC,https://youtu.be/35Pdoyi6ZoQ,35Pdoyi6ZoQ,UCv83tO5cePwHMt1952IVVHw,35Pdoyi6ZoQ-t15.84,we've essentially covered what you can see on ...,15.84,20.60


In [4]:
df.shape

(208619, 9)

In [5]:
df['title'].value_counts()

title
Building a machine learning app | 10 hour coding livestream                                       4587
Learn TensorFlow and Deep Learning fundamentals with Python (code-first introduction) Part 2/2    3608
2020 Machine Learning Roadmap (95% valid for 2022)                                                2754
Machine Learning Holiday Live Stream                                                              1760
Building a Deep Learning PC                                                                       1389
                                                                                                  ... 
I&#39;m at ICML19 :)                                                                                 9
Apple or iPod??? Easy Fix for Adversarial Textual Attacks on OpenAI&#39;s CLIP Model! #Shorts        9
Functional Friday | Raw Training Australia                                                           8
Did I crash the NFT market?                                        

In [6]:
df.columns

Index(['title', 'published', 'url', 'video_id', 'channel_id', 'id', 'text',
       'start', 'end'],
      dtype='str')

In [1]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="qwen3-embedding:8b"
)

vector = embeddings.embed_query("machine learning roadmap")

print("Embedding generated:", len(vector) > 0)
print("Vector dimensions:", len(vector))
print("First 5 values:", vector[:5])

Embedding generated: True
Vector dimensions: 4096
First 5 values: [0.043601114, 0.006055383, -0.03062202, -0.022398222, 0.06784205]


In [2]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model="qwen3-embedding:8b")

docs = [
    Document(page_content="A machine learning roadmap covers statistics, Python, and ML algorithms."),
    Document(page_content="Deep learning uses neural networks to learn complex patterns."),
    Document(page_content="RAG combines document retrieval with large language models.")
]

db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="embedding_test"
)

results = db.similarity_search("How should I learn machine learning?", k=2)

for doc in results:
    print(doc.page_content)

A machine learning roadmap covers statistics, Python, and ML algorithms.
Deep learning uses neural networks to learn complex patterns.


In [3]:
import os
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

DB_DIR = "chroma_store"

embeddings = OllamaEmbeddings(model="qwen3-embedding:8b")

print("Database directory exists:", os.path.exists(DB_DIR))

if os.path.exists(DB_DIR):
    db = Chroma(
        persist_directory=DB_DIR,
        embedding_function=embeddings
    )

    print("Collection count:", db._collection.count())
    print("Collection name:", db._collection.name)

    sample = db.get(limit=3)

    for i, text in enumerate(sample["documents"] or []):
        print(f"\nDocument {i + 1}:")
        print(text[:300])

        if sample["metadatas"]:
            print("Metadata:", sample["metadatas"][i])

    query_results = db.similarity_search(
        "machine learning roadmap",
        k=5
    )

    print("\nRetrieved documents:", len(query_results))

    for doc in query_results:
        print(doc.page_content[:200])
else:
    print("Chroma database does not exist.")

Database directory exists: True
Collection count: 0
Collection name: langchain

Retrieved documents: 0
